# PH-05 Classification - AirGlobal Dataset

Phạm vi xử lý:

- Đọc output từ PH-03: `outputs/ph03/processed_airglobal_features.pkl`
- Không đọc raw dataset
- Không đọc hoặc tạo datalake
- Không dùng trực tiếp `aqi`, `aqi_bucket` hoặc các cột target làm feature để tránh data leakage
- Train các mô hình baseline và boosting
- Tùy chọn bật Optuna và Stacking Ensemble
- Lưu metrics, biểu đồ, model và metadata vào `outputs/ph05/` và `models/`

Chạy full bằng cách chỉnh các biến cấu hình trong section 2:

- `MAX_TRAIN_ROWS = 0`
- `MAX_VAL_ROWS = 0`
- `MAX_TEST_ROWS = 0`
- `OPTUNA_TRIALS = 20` hoặc `50`
- `TRAIN_STACKING = True`

## 1. Import thư viện

Cell này nạp các thư viện cần thiết cho training, evaluation, visualization và lưu model.
Các thư viện như XGBoost, LightGBM, Optuna, SHAP là optional. Nếu máy chưa cài, notebook vẫn chạy với các model sklearn.

In [ ]:
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

## 2. Cấu hình đường dẫn và chế độ chạy

Notebook này giả định bạn chạy từ thư mục gốc project hoặc từ thư mục `notebooks/`.

Nếu muốn chạy full dataset, đặt các biến `MAX_*_ROWS = 0`.
Nếu muốn chạy nhanh để test, giữ giới hạn số dòng như bên dưới.

In [ ]:
def find_project_root() -> Path:
    """Tìm thư mục gốc project dựa trên sự tồn tại của folder outputs/ hoặc data/."""
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "outputs").exists() or (parent / "data").exists():
            return parent
    return current


BASE_DIR = find_project_root()

PH03_OUTPUT_DIR = BASE_DIR / "outputs" / "ph03"
PH05_OUTPUT_DIR = BASE_DIR / "outputs" / "ph05"
MODEL_DIR = BASE_DIR / "models"

PH05_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PKL = PH03_OUTPUT_DIR / "processed_airglobal_features.pkl"
INPUT_CSV = PH03_OUTPUT_DIR / "processed_airglobal_features.csv"
FEATURE_CATALOG = PH03_OUTPUT_DIR / "feature_catalog.csv"

RANDOM_STATE = 42
TARGET_COL = "target_aqi_bucket"
SPLIT_COL = "split"

# Set 0 để dùng toàn bộ dữ liệu.
MAX_TRAIN_ROWS = 50_000
MAX_VAL_ROWS = 10_000
MAX_TEST_ROWS = 10_000

# Set > 0 để bật Optuna tuning. Ví dụ: 20 hoặc 50.
OPTUNA_TRIALS = 0

# Bật Stacking nếu cần lấy kết quả cuối cho báo cáo. Chạy lâu hơn.
TRAIN_STACKING = False

print("Project root:", BASE_DIR)
print("PH-03 input folder:", PH03_OUTPUT_DIR)
print("PH-05 output folder:", PH05_OUTPUT_DIR)
print("Model folder:", MODEL_DIR)

## 3. Đọc dataset đã tiền xử lý từ PH-03

Classification không đọc raw data. Nó chỉ dùng dataset đã được PH-03 tạo ra.
Điều này giúp pipeline rõ ràng:

`raw data -> PH-03 preprocessing -> processed dataset -> PH-05 classification`

In [ ]:
def load_processed_dataset() -> pd.DataFrame:
    """Đọc processed dataset từ PH-03.

    Ưu tiên đọc pickle vì giữ được dtype tốt và nhanh hơn CSV.
    Nếu không có pickle, fallback sang CSV.
    """
    if INPUT_PKL.exists():
        df = pd.read_pickle(INPUT_PKL)
        print(f"-> Loaded processed pickle: {INPUT_PKL.relative_to(BASE_DIR)}")
    elif INPUT_CSV.exists():
        df = pd.read_csv(INPUT_CSV)
        print(f"-> Loaded processed CSV: {INPUT_CSV.relative_to(BASE_DIR)}")
    else:
        raise FileNotFoundError(
            "Không tìm thấy processed dataset. Hãy chạy notebook/script PH-03 trước."
        )

    if TARGET_COL not in df.columns:
        raise ValueError(f"Không tìm thấy target column: {TARGET_COL}")

    if SPLIT_COL not in df.columns:
        raise ValueError(f"Không tìm thấy split column: {SPLIT_COL}")

    df = df.dropna(subset=[TARGET_COL, SPLIT_COL]).copy()
    df[SPLIT_COL] = df[SPLIT_COL].astype(str)

    print(f"-> Dataset: {len(df):,} rows | {df.shape[1]} columns")
    print("-> Split distribution:")
    print(df[SPLIT_COL].value_counts())
    print("-> Target distribution:")
    print(df[TARGET_COL].value_counts())

    return df


df = load_processed_dataset()

## 4. Chọn feature và tránh data leakage

Các cột như `aqi`, `aqi_bucket`, `target_aqi_bucket` không được dùng làm feature.
Nếu dùng trực tiếp `aqi` để dự đoán `AQI Bucket`, mô hình có thể bị data leakage vì label thường được suy ra từ AQI.

In [ ]:
LEAKAGE_COLUMNS = {
    "aqi",
    "aqi_bucket",
    "aqi_bucket_clean",
    "AQI",
    "AQI_Bucket",
    TARGET_COL,
    SPLIT_COL,
}

NON_FEATURE_COLUMNS = {
    "date",
    "source_file",
}


def get_feature_columns(df: pd.DataFrame) -> Tuple[List[str], List[str], List[str]]:
    """Lấy danh sách feature từ feature_catalog nếu có, nếu không thì tự suy luận từ dataframe."""
    if FEATURE_CATALOG.exists():
        catalog = pd.read_csv(FEATURE_CATALOG)
        if "feature" in catalog.columns:
            feature_cols = [c for c in catalog["feature"].dropna().astype(str).tolist() if c in df.columns]
        else:
            feature_cols = []
    else:
        feature_cols = []

    if not feature_cols:
        excluded = LEAKAGE_COLUMNS.union(NON_FEATURE_COLUMNS)
        feature_cols = [c for c in df.columns if c not in excluded]

    safe_features = []
    leakage_lower = {c.lower() for c in LEAKAGE_COLUMNS}
    for col in feature_cols:
        lower = col.lower()
        if lower in leakage_lower:
            continue
        if "target" in lower:
            continue
        if lower in ["aqi", "aqi_value", "aqi score", "aqi_score"]:
            continue
        safe_features.append(col)

    categorical_features = [
        c for c in safe_features
        if str(df[c].dtype) in ["object", "string", "category"]
    ]
    numeric_features = [c for c in safe_features if c not in categorical_features]

    print(f"-> Numeric features: {len(numeric_features)}")
    print(f"-> Categorical features: {len(categorical_features)}")
    print(f"-> Total features: {len(safe_features)}")

    return safe_features, numeric_features, categorical_features


feature_cols, numeric_features, categorical_features = get_feature_columns(df)

## 5. Chia train, validation và test

PH-03 đã tạo cột `split`, nên PH-05 chỉ đọc lại đúng split đó.
Validation dùng để chọn model/tuning, test dùng để đánh giá cuối.

In [ ]:
def stratified_sample_indices(y: pd.Series, max_rows: int) -> List[int]:
    """Sample giữ tương đối tỷ lệ class để chạy nhanh khi chưa muốn full dataset."""
    if max_rows <= 0 or len(y) <= max_rows:
        return list(y.index)

    parts = []
    for label, idx in y.groupby(y).groups.items():
        idx_list = list(idx)
        n_label = max(1, int(round(max_rows * len(idx_list) / len(y))))
        n_label = min(n_label, len(idx_list))
        sampled = pd.Series(idx_list).sample(n_label, random_state=RANDOM_STATE).tolist()
        parts.extend(sampled)

    if len(parts) > max_rows:
        parts = pd.Series(parts).sample(max_rows, random_state=RANDOM_STATE).tolist()

    return sorted(parts)


def build_split_data(df: pd.DataFrame):
    encoder = LabelEncoder()
    encoder.fit(sorted(df[TARGET_COL].astype(str).unique()))

    data = {}
    for split_name, max_rows in [
        ("train", MAX_TRAIN_ROWS),
        ("val", MAX_VAL_ROWS),
        ("test", MAX_TEST_ROWS),
    ]:
        part = df[df[SPLIT_COL] == split_name].copy()
        y_raw = part[TARGET_COL].astype(str)
        selected_idx = stratified_sample_indices(y_raw, max_rows)
        part = part.loc[selected_idx].copy()

        X = part[feature_cols].copy()
        y = encoder.transform(part[TARGET_COL].astype(str))

        data[f"X_{split_name}"] = X
        data[f"y_{split_name}"] = y

    print(
        "-> Used rows:",
        f"train={len(data['X_train']):,},",
        f"val={len(data['X_val']):,},",
        f"test={len(data['X_test']):,}"
    )

    return data, encoder


data, label_encoder = build_split_data(df)

## 6. Tạo preprocessing transformer cho model

Numeric features được scale bằng `StandardScaler`.
Categorical features được one-hot encode.

In [ ]:
def make_one_hot_encoder():
    """Tạo OneHotEncoder tương thích nhiều phiên bản scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", make_one_hot_encoder(), categorical_features),
    ],
    remainder="drop",
)

print("-> Preprocessing transformer created.")

## 7. Khai báo mô hình

Notebook train các model baseline trước. XGBoost và LightGBM chỉ chạy nếu máy đã cài thư viện.

In [ ]:
def optional_import(module_name: str):
    try:
        return __import__(module_name)
    except Exception:
        return None


def make_pipeline(model):
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])


def build_models() -> Dict[str, Pipeline]:
    models = {
        "LogisticRegression_SGD": make_pipeline(
            SGDClassifier(
                loss="log_loss",
                alpha=0.0005,
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )
        ),
        "DecisionTree": make_pipeline(
            DecisionTreeClassifier(
                max_depth=18,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )
        ),
        "RandomForest": make_pipeline(
            RandomForestClassifier(
                n_estimators=160,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
        "ExtraTrees": make_pipeline(
            ExtraTreesClassifier(
                n_estimators=160,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
    }

    xgb = optional_import("xgboost")
    if xgb is not None:
        models["XGBoost"] = make_pipeline(
            xgb.XGBClassifier(
                n_estimators=350,
                max_depth=8,
                learning_rate=0.06,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="multi:softprob",
                eval_metric="mlogloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        )
    else:
        print("Warning: xgboost is not installed. XGBoost skipped.")

    lgb = optional_import("lightgbm")
    if lgb is not None:
        models["LightGBM"] = make_pipeline(
            lgb.LGBMClassifier(
                n_estimators=450,
                learning_rate=0.05,
                num_leaves=63,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )
        )
    else:
        print("Warning: lightgbm is not installed. LightGBM skipped.")

    return models


models = build_models()
print("-> Models:", list(models.keys()))

## 8. Train và đánh giá model

Mỗi model được train trên train set, chọn theo Macro F1 trên validation set, sau đó báo cáo test set.

In [ ]:
def evaluate_model(model_name: str, model: Pipeline, X, y, split_name: str) -> Dict[str, float]:
    pred = model.predict(X)
    precision, recall, _, _ = precision_recall_fscore_support(
        y, pred, average="macro", zero_division=0
    )
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y, pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "cohen_kappa": float(cohen_kappa_score(y, pred)),
    }


def save_confusion_matrix(model_name: str, model: Pipeline, X, y, split_name: str = "test"):
    pred = model.predict(X)
    labels = np.arange(len(label_encoder.classes_))
    cm = confusion_matrix(y, pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        title=f"Confusion Matrix - {model_name} ({split_name})",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=np.arange(len(label_encoder.classes_)),
        yticks=np.arange(len(label_encoder.classes_)),
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    fig.tight_layout()
    fig.savefig(PH05_OUTPUT_DIR / f"confusion_matrix_{model_name}_{split_name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


metrics_rows = []
trained_models: Dict[str, Pipeline] = {}

print("4. Training baseline and ensemble models...")

for name, model in models.items():
    print(f"-> Training {name}")
    model.fit(data["X_train"], data["y_train"])
    trained_models[name] = model

    metrics_rows.append(evaluate_model(name, model, data["X_val"], data["y_val"], "val"))
    metrics_rows.append(evaluate_model(name, model, data["X_test"], data["y_test"], "test"))

    save_confusion_matrix(name, model, data["X_test"], data["y_test"], "test")

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])

## 9. Optional: Optuna tuning

Cell này chỉ chạy nếu `OPTUNA_TRIALS > 0` và máy có cài Optuna cùng XGBoost.
Nếu không cần tuning, có thể bỏ qua cell này.

In [ ]:
def tune_xgboost_with_optuna() -> Optional[Pipeline]:
    if OPTUNA_TRIALS <= 0:
        print("-> Optuna disabled. Set OPTUNA_TRIALS > 0 to tune.")
        return None

    optuna = optional_import("optuna")
    xgb = optional_import("xgboost")
    if optuna is None or xgb is None:
        print("Warning: optuna or xgboost is not installed. Tuning skipped.")
        return None

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 700),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "objective": "multi:softprob",
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        pipe = make_pipeline(xgb.XGBClassifier(**params))
        pipe.fit(data["X_train"], data["y_train"])
        pred = pipe.predict(data["X_val"])
        return f1_score(data["y_val"], pred, average="macro", zero_division=0)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

    best_params = dict(study.best_params)
    best_params.update({
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    })

    best_model = make_pipeline(xgb.XGBClassifier(**best_params))
    best_model.fit(data["X_train"], data["y_train"])

    with open(PH05_OUTPUT_DIR / "optuna_xgboost_best_params.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2)

    print("-> Tuned XGBoost validation Macro F1:", study.best_value)
    return best_model


tuned_xgb = tune_xgboost_with_optuna()

if tuned_xgb is not None:
    trained_models["XGBoost_Optuna"] = tuned_xgb
    new_rows = [
        evaluate_model("XGBoost_Optuna", tuned_xgb, data["X_val"], data["y_val"], "val"),
        evaluate_model("XGBoost_Optuna", tuned_xgb, data["X_test"], data["y_test"], "test"),
    ]
    metrics_df = pd.concat([metrics_df, pd.DataFrame(new_rows)], ignore_index=True)
    metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])

## 10. Optional: Stacking Ensemble

Stacking thường chạy lâu hơn vì phải train nhiều base models.
Nếu `TRAIN_STACKING = False`, cell này sẽ bỏ qua.

In [ ]:
def train_stacking_if_enabled() -> Optional[Pipeline]:
    if not TRAIN_STACKING:
        print("-> Stacking disabled. Set TRAIN_STACKING = True to train.")
        return None

    estimators = []

    if "XGBoost_Optuna" in trained_models:
        estimators.append(("xgb", trained_models["XGBoost_Optuna"]))
    elif "XGBoost" in trained_models:
        estimators.append(("xgb", trained_models["XGBoost"]))

    if "LightGBM" in trained_models:
        estimators.append(("lgbm", trained_models["LightGBM"]))

    if "RandomForest" in trained_models:
        estimators.append(("rf", trained_models["RandomForest"]))

    if "ExtraTrees" in trained_models:
        estimators.append(("extra", trained_models["ExtraTrees"]))

    if len(estimators) < 2:
        print("Warning: Not enough base models for stacking.")
        return None

    meta_learner = LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_learner,
        stack_method="predict_proba",
        cv=3,
        n_jobs=-1,
        passthrough=False,
    )

    print("-> Training Stacking Ensemble")
    stack.fit(data["X_train"], data["y_train"])
    return stack


stacking_model = train_stacking_if_enabled()

if stacking_model is not None:
    trained_models["StackingEnsemble"] = stacking_model
    new_rows = [
        evaluate_model("StackingEnsemble", stacking_model, data["X_val"], data["y_val"], "val"),
        evaluate_model("StackingEnsemble", stacking_model, data["X_test"], data["y_test"], "test"),
    ]
    metrics_df = pd.concat([metrics_df, pd.DataFrame(new_rows)], ignore_index=True)
    metrics_df.to_csv(PH05_OUTPUT_DIR / "model_metrics.csv", index=False, encoding="utf-8-sig")
    save_confusion_matrix("StackingEnsemble", stacking_model, data["X_test"], data["y_test"], "test")

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])

## 11. Chọn best model theo validation Macro F1

Validation set dùng để chọn mô hình tốt nhất.
Sau đó báo cáo kết quả cuối trên test set.

In [ ]:
val_metrics = metrics_df[metrics_df["split"] == "val"].copy()
best_model_name = val_metrics.sort_values("macro_f1", ascending=False).iloc[0]["model"]
best_model = trained_models[best_model_name]

print("-> Best model by validation Macro F1:", best_model_name)

test_metrics = metrics_df[metrics_df["split"] == "test"].sort_values("macro_f1", ascending=False)
test_metrics

## 12. Feature importance và explainability

Ưu tiên dùng feature importance của tree-based model nếu có.
Nếu không có, fallback sang permutation importance.

In [ ]:
def get_transformed_feature_names(model: Pipeline) -> List[str]:
    pre = model.named_steps.get("preprocess")
    if pre is None:
        return feature_cols

    try:
        return list(pre.get_feature_names_out())
    except Exception:
        names = []
        names.extend([f"num__{c}" for c in numeric_features])
        names.extend([f"cat__{c}" for c in categorical_features])
        return names


def save_feature_importance(model_name: str, model: Pipeline) -> pd.DataFrame:
    model_step = model.named_steps.get("model") if isinstance(model, Pipeline) else model
    feature_names = get_transformed_feature_names(model) if isinstance(model, Pipeline) else feature_cols

    if hasattr(model_step, "feature_importances_"):
        importances = np.asarray(model_step.feature_importances_)
        n = min(len(importances), len(feature_names))
        result = pd.DataFrame({
            "feature": feature_names[:n],
            "importance": importances[:n],
        }).sort_values("importance", ascending=False)
    else:
        print("-> Using permutation importance fallback.")
        perm = permutation_importance(
            model,
            data["X_test"],
            data["y_test"],
            n_repeats=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            scoring="f1_macro",
        )
        result = pd.DataFrame({
            "feature": feature_cols,
            "importance": perm.importances_mean,
        }).sort_values("importance", ascending=False)

    result.to_csv(PH05_OUTPUT_DIR / "best_model_feature_importance.csv", index=False, encoding="utf-8-sig")

    top20 = result.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top20["feature"], top20["importance"])
    ax.set_title(f"Top 20 Feature Importance - {model_name}")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    fig.savefig(PH05_OUTPUT_DIR / "best_model_feature_importance_top20.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return result


feature_importance_df = save_feature_importance(best_model_name, best_model)
feature_importance_df.head(20)

## 13. Optional: SHAP summary

SHAP có thể chạy lâu và tùy thuộc model. Nếu lỗi, cell sẽ bỏ qua và vẫn giữ feature importance fallback.

In [ ]:
def try_save_shap_summary(model_name: str, model: Pipeline):
    shap = optional_import("shap")
    if shap is None:
        print("Warning: shap is not installed. SHAP skipped.")
        return

    try:
        if not isinstance(model, Pipeline):
            print("Warning: SHAP currently expects a sklearn Pipeline. Skipped.")
            return

        pre = model.named_steps["preprocess"]
        model_step = model.named_steps["model"]

        X_sample = data["X_test"].sample(min(1500, len(data["X_test"])), random_state=RANDOM_STATE)
        X_transformed = pre.transform(X_sample)
        feature_names = get_transformed_feature_names(model)

        explainer = shap.TreeExplainer(model_step)
        shap_values = explainer.shap_values(X_transformed)

        plt.figure()
        shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(PH05_OUTPUT_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
        plt.close()
        print("-> SHAP summary saved.")
    except Exception as exc:
        print(f"Warning: SHAP skipped due to {type(exc).__name__}: {exc}")


try_save_shap_summary(best_model_name, best_model)

## 14. Lưu production artifacts

Các file quan trọng:

- `best_aqi_classifier_pipeline.pkl`: pipeline tốt nhất gồm preprocessing transformer + model
- `label_encoder.pkl`: mã hóa/giải mã label
- `feature_columns.json`: danh sách feature đầu vào
- `model_metadata.json`: thông tin cấu hình và kết quả

In [ ]:
with open(MODEL_DIR / "best_aqi_classifier_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open(MODEL_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

with open(MODEL_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

metadata = {
    "best_model": best_model_name,
    "target_col": TARGET_COL,
    "split_col": SPLIT_COL,
    "n_features": len(feature_cols),
    "n_numeric_features": len(numeric_features),
    "n_categorical_features": len(categorical_features),
    "train_rows": int(len(data["X_train"])),
    "val_rows": int(len(data["X_val"])),
    "test_rows": int(len(data["X_test"])),
    "optuna_trials": int(OPTUNA_TRIALS),
    "train_stacking": bool(TRAIN_STACKING),
    "classes": list(label_encoder.classes_),
}

with open(PH05_OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("-> Saved production artifacts.")
print("Model folder:", MODEL_DIR.relative_to(BASE_DIR))
print("Output folder:", PH05_OUTPUT_DIR.relative_to(BASE_DIR))

## 15. Tóm tắt kết quả

Cell này in nhanh các file output và bảng test metrics để tiện kiểm tra trước khi push.
Không nên push các file quá 100MB lên GitHub.

In [ ]:
print("Output files:")
for path in sorted(PH05_OUTPUT_DIR.glob("*")):
    print("-", path.relative_to(BASE_DIR))

print("Model files:")
for path in sorted(MODEL_DIR.glob("*")):
    print("-", path.relative_to(BASE_DIR), f"({path.stat().st_size / (1024 * 1024):.2f} MB)")

print("Final test metrics:")
metrics_df[metrics_df["split"] == "test"].sort_values("macro_f1", ascending=False)